In [0]:
%pip install uv
dbutils.library.restartPython()

In [0]:
%sh
uv pip install -r requirements.txt
uv pip install -e '.[dev]'

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "pixels", "Schema")
dbutils.widgets.text("target", "dev", "Target")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TARGET = dbutils.widgets.get("target")
print(f"catalog={CATALOG}, schema={SCHEMA}, target={TARGET}")

In [0]:
%sh
mkdir -p dist
python -m build -w -o dist
echo "Built: $(ls dist/databricks_pixels*.whl 2>/dev/null | tail -1)"

In [0]:
%sh
# Build OHIF tarball (if present and non-trivial)
if [ -d apps/dicom-web/ohif ] && [ "$(ls apps/dicom-web/ohif/ | wc -l)" -gt 5 ]; then
    tar czf dist/ohif.tar.gz -C apps/dicom-web ohif
    echo "Created dist/ohif.tar.gz ($(du -sh dist/ohif.tar.gz | cut -f1))"
fi

# Build vista3d tarball (if models/monai present)
# Uses --transform instead of BSD tar's -s flag (GNU tar on Linux)
if [ -d models/monai ]; then
    tar czf dist/vista3d.tar.gz \
        --exclude='model' --exclude='assets' \
        --transform 's|^monai/|vista3d/|' --transform 's|^monai$|vista3d|' \
        -C models monai
    echo "Created dist/vista3d.tar.gz ($(du -sh dist/vista3d.tar.gz | cut -f1))"
fi

In [0]:
%python
import subprocess, os
from pathlib import Path
from databricks.sdk import WorkspaceClient

# Render dashboard — inline using WorkspaceClient in-process;
def _cloud_suffix(host: str) -> str:
    return "azure" if ".azuredatabricks.net" in host else ("gcp" if ".gcp.databricks.com" in host else "aws")

_w = WorkspaceClient()
_viewer_host = f"https://pixels-dicomweb-{_w.get_workspace_id()}.{_cloud_suffix(_w.config.host)}.databricksapps.com"
_out = Path(f"./dist/Pixels Object Catalog dashboard.lvdash.json")
_out.parent.mkdir(parents=True, exist_ok=True)
_out.write_text(
    Path(f"./ai-bi/Pixels Object Catalog dashboard.lvdash.json.tmpl")
    .read_text()
    .replace("__CATALOG__", CATALOG)
    .replace("__SCHEMA__", SCHEMA)
    .replace("__VIEWER_HOST__", _viewer_host)
)
print(f"Rendered {_out} | catalog={CATALOG} schema={SCHEMA} viewer_host={_viewer_host}")

print(f"""\n====== Run from the web terminal ======\n
databricks bundle deploy -t {TARGET} --auto-approve --var catalog={CATALOG} --var schema={SCHEMA}\n
=======================================""")